# Benchmark State Preparation

Quantum state preparation is the task of building a quantum circuit that takes the default initial state $|0^{\otimes n}\rangle$ into a desired target state $|\psi\rangle$.
Many algorithms assume that some specific state is available (e.g. a superposition, a classical distribution, or an approximate eigenstate), therefore state preparation is a basic primitive in quantum algorithms.


Given a set of amplitudes, $\{a_j\}$, we benchmark the preparation of the quantum state
$$
|\psi\rangle = \sum_{j=0}^{2^n-1} a_j |j\rangle~.
$$

***

### The model
We prepare a state with linear amplitudes
$$|\psi\rangle = \frac{1}{\sqrt{M}}\sum_{x=0}^{2^n-1} x |x\rangle~, $$
where $M=\sum_{x} x$ is the normalization constant.


### The Scoring function
The score is defined in terms of the **total variation distance**
 $$D_{TV}(p,u) = \frac{1}{2}\sum_{i=0}^{n-1}\left|p_i-u_i \right|~~,$$ 
where $p_i$ and $u_i$ are the observed and expected propbabilities, correspondingly.
The distance is bounded between zero and $1-1/n$. To obtain a score between zero and one (one is a perfect result) we define the score as 

$$S =1 - \frac{D_{TV}(p,u)}{1-1/n} ~~.$$

***
***

## Defining a `BenchmarkExample` for Linear State Preparation

In [1]:
import asyncio
import datetime
import sys

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES

from classiq import *

In [2]:
# Define a utility function for amplitude distribution


def linear_amplitudes_probabilities(n):
    """
    Returns the probabilities of the linear amplitudes.
    """
    N = 2**n
    amps = np.arange(N)
    amps = amps / np.linalg.norm(amps)
    return np.abs(amps) ** 2

In [3]:
# How to construct main for a given number of qubits


class SPExample(BenchmarkExample):
    def __init__(self, num_qubits: int):
        super().__init__(
            name="state_preparation",
            num_qubits=num_qubits,
        )

    def create_main(self) -> callable:
        @qfunc
        def main(x: Output[QNum[self.num_qubits]]):
            allocate(x)
            prepare_linear_amplitudes(x)

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_sample()
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()
        df = result[0].value.dataframe

        n = self.num_qubits
        N = 2**n

        # Build measured distribution p over all N states
        p = np.zeros(N)
        # Assuming the dataframe uses "x" as the state index
        p[df["x"]] = df["probability"].to_numpy()

        u = linear_amplitudes_probabilities(self.num_qubits)
        d_tv = 0.5 * np.sum(np.abs(p - u))
        score = 1.0 - d_tv / (1.0 - 1.0 / n)

        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": float(score),
            "execution_time": exec_minutes,
        }

***
***

## Benchmarking a 4-qubits state preperation

Define a specific example on 4 qubits:

In [4]:
example = SPExample(num_qubits=4)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3B8FBOGvqoBf5a564zuRdqbDQST


Define the number of shots and other hyperparameters:

In [5]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

### Choosing on which backend to run 

Define `HardwareRunner` for each backend. Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited after defining the `ResultCollector`.)*

In [6]:
classiq_runners = [
    HardwareRunner("Classiq", backend_name, **common_config)
    for backend_name in HARDWARES["Classiq"][0:2]
]

ionq_runners = [
    HardwareRunner("IonQ", backend_name, **common_config)
    for backend_name in HARDWARES["IonQ"][0:1]
]


ibm_runners = [
    HardwareRunner(
        "IBM Quantum",
        backend_name,
        **common_config,
        backend_kwargs={
            "access_token": "PUT YOUR TOKEN HERE",
            "channel": "PUT YOUR CHANNEL HERE",
            "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
        },
    )
    for backend_name in HARDWARES["IBM Quantum"][0:1]
]

runners = [
    *classiq_runners,
    *ionq_runners,
    *ibm_runners,
]

### Set a `ResultCollector` with a file name fixed for the current example

In [7]:
FILENAME = f"data/{example.name}{example.num_qubits}.csv"

In [8]:
collector = ResultCollector(FILENAME, build_each_time=True)

In [9]:
print(
    "=" * 10
    + f"  {example.name}-{example.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  state_preparation-4 (2026-03-18 21:37:35.565714)   ==========
2026-03-18 21:37:40.894660: Submit state_preparation-4 for Classiq - simulator
2026-03-18 21:37:43.016150: Completed state_preparation-4 for Classiq - simulator --> Score {'score': 0.9479569892473119, 'execution_time': 0.010557266666666667}
** Report updated: state_preparation-4 for Classiq - simulator


In [10]:
await collector.print_status()

========== (2026-03-18 21:37:43.857105)   ==========
state_preparation-4 | IonQ - qpu.forte-1 | COMPLETED | score=0.8666 | time=10.14 min
state_preparation-4 | Classiq - simulator_statevector | COMPLETED | score=1.0000 | time=0.01 min
state_preparation-4 | IBM Quantum - ibm_kingston | COMPLETED | score=0.8535 | time=479.33 min
state_preparation-4 | Classiq - simulator | COMPLETED | score=0.9480 | time=0.01 min
